# ProteinLens — Day 1: Dataset EDA

**Goal:** Load Food-101, filter to 20 high-protein classes, visualise sample images, and verify class distribution.

**Dataset:** [Food-101](https://www.kaggle.com/datasets/dansbecker/food-101) — 101 food categories, 1000 images each (750 train / 250 test).

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
import random

print('Libraries loaded ✓')

## 1. Define our 20 high-protein classes

Food-101 class names use underscores and match the folder names exactly.

In [ ]:
# These are the Food-101 class names that map to our high-protein targets
# Note: Food-101 doesn't have every class — we map to closest available
PROTEIN_CLASSES = [
    'chicken_wings',      # → chicken breast (closest in Food-101)
    'eggs_benedict',      # → egg
    'grilled_salmon',     # → salmon
    'filet_mignon',       # → steak
    'tuna_tartare',       # → tuna
    'shrimp_and_grits',   # → shrimp
    'greek_salad',        # → includes feta / protein-rich
    'lobster_bisque',     # → lobster
    'edamame',            # → edamame ✓ exact match
    'lentil_soup',        # → lentils
    'pulled_pork_sandwich', # → pork
    'beef_carpaccio',     # → beef
    'crab_cakes',         # → crab
    'lobster_roll_sandwich', # → lobster
    'hamburger',          # → beef burger ✓
    'pork_chop',          # → pork chop ✓ exact
    'baby_back_ribs',     # → pork ribs
    'steak',              # → steak ✓ exact
    'sashimi',            # → fish/protein
    'oysters',            # → seafood protein
]

# Cleaner display names for our app
CLASS_DISPLAY_NAMES = {
    'chicken_wings': 'Chicken',
    'eggs_benedict': 'Egg',
    'grilled_salmon': 'Salmon',
    'filet_mignon': 'Steak',
    'tuna_tartare': 'Tuna',
    'shrimp_and_grits': 'Shrimp',
    'greek_salad': 'Greek Salad',
    'lobster_bisque': 'Lobster',
    'edamame': 'Edamame',
    'lentil_soup': 'Lentils',
    'pulled_pork_sandwich': 'Pork',
    'beef_carpaccio': 'Beef',
    'crab_cakes': 'Crab',
    'lobster_roll_sandwich': 'Lobster Roll',
    'hamburger': 'Burger',
    'pork_chop': 'Pork Chop',
    'baby_back_ribs': 'Ribs',
    'steak': 'Steak',
    'sashimi': 'Sashimi',
    'oysters': 'Oysters',
}

print(f'Target classes: {len(PROTEIN_CLASSES)}')
for i, c in enumerate(PROTEIN_CLASSES):
    print(f'  {i+1:02d}. {c} → "{CLASS_DISPLAY_NAMES[c]}')

## 2. Load dataset

**Option A (recommended):** Download via `tensorflow_datasets` — handles splitting automatically.

**Option B:** Download from Kaggle and point `DATA_DIR` at the unzipped folder.

Run the cell below to install and download. First run takes ~5 min on Colab.

In [ ]:
# Option A: tensorflow_datasets (easiest on Colab)
# Uncomment and run:

# !pip install tensorflow-datasets -q
# import tensorflow_datasets as tfds
# ds_info = tfds.load('food101', with_info=True)

# Option B: Kaggle download
# 1. Upload kaggle.json to Colab
# 2. Run:
# !pip install kaggle -q
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d dansbecker/food-101 --unzip -p ml/data/

# Set this to your local data path:
DATA_DIR = Path('data/food-101/images')   # adjust if needed

print('Data directory set to:', DATA_DIR)

In [ ]:
def count_images(data_dir: Path, classes: list) -> dict:
    """Count images per class in the dataset."""
    counts = {}
    missing = []
    for cls in classes:
        cls_path = data_dir / cls
        if cls_path.exists():
            counts[cls] = len(list(cls_path.glob('*.jpg')))
        else:
            missing.append(cls)
            counts[cls] = 0
    if missing:
        print(f'⚠️  Missing classes: {missing}')
    return counts

# Only run if data is downloaded
if DATA_DIR.exists():
    counts = count_images(DATA_DIR, PROTEIN_CLASSES)
    total = sum(counts.values())
    print(f'Total images across {len(PROTEIN_CLASSES)} classes: {total:,}')
    for cls, n in counts.items():
        bar = '█' * (n // 50)
        print(f'  {cls:<30} {n:>5} {bar}')
else:
    print('Data not yet downloaded — run the download cell above first.')
    print('Expected: ~1000 images per class = ~20,000 total')

## 3. Visualise sample images per class

In [ ]:
def plot_class_samples(data_dir: Path, classes: list, n_per_class: int = 4):
    """Plot a grid of sample images from each class."""
    n_classes = len(classes)
    fig, axes = plt.subplots(n_classes, n_per_class, figsize=(n_per_class * 2.5, n_classes * 2.5))
    fig.suptitle('ProteinLens — Sample Images per Class', fontsize=14, fontweight='bold', y=1.01)

    for row, cls in enumerate(classes):
        cls_path = data_dir / cls
        if not cls_path.exists():
            for col in range(n_per_class):
                axes[row, col].axis('off')
                axes[row, col].set_title('Missing', fontsize=8)
            continue

        images = list(cls_path.glob('*.jpg'))
        samples = random.sample(images, min(n_per_class, len(images)))

        for col, img_path in enumerate(samples):
            img = Image.open(img_path).convert('RGB').resize((224, 224))
            axes[row, col].imshow(img)
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(
                    CLASS_DISPLAY_NAMES.get(cls, cls),
                    fontsize=9, rotation=0, labelpad=60, va='center'
                )

    plt.tight_layout()
    plt.savefig('../assets/eda_samples.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved to assets/eda_samples.png')

if DATA_DIR.exists():
    plot_class_samples(DATA_DIR, PROTEIN_CLASSES)
else:
    print('Download data first to visualise samples.')

## 4. Protein content overview

In [ ]:
with open('protein_data.json') as f:
    protein_db = json.load(f)

# Remove metadata key
foods = {k: v for k, v in protein_db.items() if k != '_metadata'}

labels = [v['label'] for v in foods.values()]
proteins = [v['protein_g'] for v in foods.values()]
calories = [v['calories'] for v in foods.values()]

# Sort by protein descending
sorted_idx = np.argsort(proteins)[::-1]
labels_s = [labels[i] for i in sorted_idx]
proteins_s = [proteins[i] for i in sorted_idx]
calories_s = [calories[i] for i in sorted_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Protein bar chart
colors = ['#2ecc71' if p >= 20 else '#f39c12' if p >= 12 else '#e74c3c' for p in proteins_s]
bars = ax1.barh(labels_s, proteins_s, color=colors, edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Protein (g per 100g)', fontsize=11)
ax1.set_title('Protein Content by Food Class', fontsize=13, fontweight='bold')
ax1.axvline(20, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='20g threshold')
ax1.legend(fontsize=9)
for bar, val in zip(bars, proteins_s):
    ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val}g',
             va='center', fontsize=8.5)
ax1.invert_yaxis()

# Protein vs Calories scatter
scatter_colors = ['#2ecc71' if p >= 20 else '#f39c12' if p >= 12 else '#e74c3c' for p in proteins]
ax2.scatter(calories, proteins, c=scatter_colors, s=80, edgecolors='white', linewidths=0.5, zorder=3)
for i, label in enumerate(labels):
    ax2.annotate(label, (calories[i], proteins[i]),
                 textcoords='offset points', xytext=(5, 3), fontsize=7, alpha=0.8)
ax2.set_xlabel('Calories (per 100g)', fontsize=11)
ax2.set_ylabel('Protein (g per 100g)', fontsize=11)
ax2.set_title('Protein vs Calories', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='High (≥20g)'),
    Patch(facecolor='#f39c12', label='Medium (12-20g)'),
    Patch(facecolor='#e74c3c', label='Lower (<12g)'),
]
ax2.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.savefig('../assets/eda_protein_chart.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to assets/eda_protein_chart.png')

# Summary stats
print(f'\n--- Protein Summary ---')
print(f'Max:  {max(proteins):.1f}g  ({labels[np.argmax(proteins)]})')
print(f'Min:  {min(proteins):.1f}g  ({labels[np.argmin(proteins)]})')
print(f'Mean: {np.mean(proteins):.1f}g')
print(f'Classes with ≥20g protein: {sum(1 for p in proteins if p >= 20)}/{len(proteins)}')

## 5. Day 1 Summary

| Item | Status |
|---|---|
| GitHub repo created | ✅ |
| Folder structure set up | ✅ |
| `protein_data.json` with USDA values | ✅ |
| `requirements.txt` | ✅ |
| Food-101 downloaded | ✅ |
| 20 protein classes identified | ✅ |
| EDA notebook complete | ✅ |

**Tomorrow (Day 2):** Fine-tune MobileNetV2 on our 20 classes using `train.ipynb`.